# Analisis dan Pengujian FFNN
Dataset: **Global Student Placement & Salary Dataset**

Analisis ini meliputi pengujian hyperparameter, fungsi aktivasi, learning rate, regularisasi, dan perbandingan dengan library Scikit-Learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
import sys
import os
from pathlib import Path

project_root = Path.cwd().resolve()
for candidate in [project_root, project_root / 'src', project_root.parent, project_root.parent / 'src']:
    sys.path.append(str(candidate))

from ffnn import FFNN, Layer
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
EXPERIMENT_EPOCHS = 20
BATCH_SIZE = 64
BASE_LEARNING_RATE = 0.01


## 1. Data Loading & Preprocessing

In [ ]:
from pathlib import Path

project_root = Path.cwd().resolve()
dataset_candidates = [
    project_root / 'datasetml_2026.csv',
    project_root / 'data' / 'datasetml_2026.csv',
    project_root.parent / 'datasetml_2026.csv',
    project_root.parent / 'data' / 'datasetml_2026.csv',
]

for candidate in dataset_candidates:
    if candidate.exists():
        dataset_path = candidate
        break
else:
    raise FileNotFoundError('datasetml_2026.csv not found in project root/data or parent/data directories.')

df = pd.read_csv(dataset_path)
print(f"Dataset loaded from: {dataset_path}")
print("Dataset shape:", df.shape)

# Target
y = df['placement_status'].apply(lambda x: 1 if x == 'Placed' else 0).values

# Features
X_raw = df.drop(columns=['placement_status'])
X_numeric = X_raw.select_dtypes(include=['float64', 'int64']).copy()
X_categorical = X_raw.select_dtypes(include=['object']).copy()

# One-hot encode categorical
X_cat_encoded = pd.get_dummies(X_categorical, drop_first=True)

# Combine features
X_combined = pd.concat([X_numeric, X_cat_encoded], axis=1).values

# Split 70:15:15 (train:validation:test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_combined,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

# StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_train = y_train.reshape(-1, 1)
y_val = y_val.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print(f"Feature count after preprocessing: {X_combined.shape[1]}")
print(f"Train shapes      : X={X_train.shape}, y={y_train.shape}")
print(f"Validation shapes : X={X_val.shape}, y={y_val.shape}")
print(f"Test shapes       : X={X_test.shape}, y={y_test.shape}")
print(f"Split sizes       : train={X_train.shape[0]}, val={X_val.shape[0]}, test={X_test.shape[0]}")


## Helper Functions
Fungsi untuk memplot *training* dan *validation loss* serta menampilkan metrik evaluasi.

In [ ]:
def plot_loss(history, title="Training History"):
    plt.figure(figsize=(8, 5))
    plt.plot(history['loss'], label='Train Loss')
    if 'val_loss' in history and len(history['val_loss']) > 0 and history['val_loss'][0] is not None:
        plt.plot(history['val_loss'], label='Val Loss')
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid()
    plt.show()

def evaluate_model(y_true, y_pred_proba, title="Evaluation"):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred_proba = np.asarray(y_pred_proba).reshape(-1)
    y_pred_proba = np.clip(y_pred_proba, 1e-15, 1 - 1e-15)

    y_pred_class = (y_pred_proba >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_class)
    f1 = f1_score(y_true, y_pred_class, zero_division=0)
    ll = log_loss(y_true, y_pred_proba)

    print(f"--- {title} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Log Loss : {ll:.4f}")
    print("-" * (8 + len(title)))

def fit_ffnn(net, learning_rate=BASE_LEARNING_RATE, epochs=EXPERIMENT_EPOCHS, l1_lambda=0, l2_lambda=0, verbose=0):
    # Seed diset ulang
    np.random.seed(RANDOM_SEED)
    net.compile(loss='binary_crossentropy', l1_lambda=l1_lambda, l2_lambda=l2_lambda)
    history = net.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        learning_rate=learning_rate,
        validation_data=(X_val, y_val),
        verbose=verbose,
    )
    return history


## 2. Analisis Hyperparameter
### 2.a Pengaruh Width (Depth Tetap)

Depth hidden layer dibuat tetap **2 layer**, lalu width diuji pada **8, 16, 32** neuron.

### 2.b Pengaruh Depth (Width Tetap)

Width hidden layer dibuat tetap **16 neuron**, lalu depth diuji pada **1, 2, 3** hidden layer.


In [ ]:
input_size = X_train.shape[1]

# Width experiment (depth tetap)
fixed_depth = 2
widths = [8, 16, 32]

for w in widths:
    print()
    print(f"Training model with Width {w} (Depth fixed at {fixed_depth})")
    net = FFNN()

    net.add(Layer(input_size, w, activation='relu', weight_init='random_normal', seed=RANDOM_SEED))
    for layer_idx in range(fixed_depth - 1):
        net.add(Layer(w, w, activation='relu', weight_init='random_normal', seed=RANDOM_SEED + layer_idx + 1))
    net.add(Layer(w, 1, activation='sigmoid', weight_init='random_normal', seed=RANDOM_SEED + 99))

    history = fit_ffnn(net)
    plot_loss(history, title=f"Width {w}, Depth {fixed_depth}")
    y_pred_val = net.predict(X_val)
    evaluate_model(y_val, y_pred_val, f"Width {w}, Depth {fixed_depth} (Validation)")

# Depth experiment (width tetap)
fixed_width = 16
depths = [1, 2, 3]

for d in depths:
    print()
    print(f"Training model with Depth {d} (Width fixed at {fixed_width})")
    net = FFNN()

    net.add(Layer(input_size, fixed_width, activation='relu', weight_init='random_normal', seed=RANDOM_SEED))
    for layer_idx in range(d - 1):
        net.add(Layer(fixed_width, fixed_width, activation='relu', weight_init='random_normal', seed=RANDOM_SEED + layer_idx + 1))
    net.add(Layer(fixed_width, 1, activation='sigmoid', weight_init='random_normal', seed=RANDOM_SEED + 99))

    history = fit_ffnn(net)
    plot_loss(history, title=f"Depth {d}, Width {fixed_width}")
    y_pred_val = net.predict(X_val)
    evaluate_model(y_val, y_pred_val, f"Depth {d}, Width {fixed_width} (Validation)")


### 2.c Pengaruh Fungsi Aktivasi Hidden Layer

Menguji aktivasi hidden layer: **ReLU, Sigmoid, Tanh, Linear, Leaky ReLU, ELU**.
Output layer tetap Sigmoid untuk klasifikasi biner.


In [ ]:
activation_settings = [
    ('relu', {}, 'ReLU'),
    ('sigmoid', {}, 'Sigmoid'),
    ('tanh', {}, 'Tanh'),
    ('linear', {}, 'Linear'),
    ('leaky_relu', {'alpha': 0.01}, 'Leaky ReLU'),
    ('elu', {'alpha': 1.0}, 'ELU'),
]

input_size = X_train.shape[1]

for act_name, act_kwargs, label in activation_settings:
    print()
    print(f"Training model with Activation {label}")
    net = FFNN()
    net.add(Layer(input_size, 16, activation=act_name, weight_init='random_normal', seed=RANDOM_SEED, **act_kwargs))
    net.add(Layer(16, 16, activation=act_name, weight_init='random_normal', seed=RANDOM_SEED + 1, **act_kwargs))
    net.add(Layer(16, 1, activation='sigmoid', weight_init='random_normal', seed=RANDOM_SEED + 2))

    history = fit_ffnn(net)

    plot_loss(history, title=f"Activation {label}")
    y_pred_val = net.predict(X_val)
    evaluate_model(y_val, y_pred_val, f"Activation {label} (Validation)")
    print(f"Plotting Weights and Gradients distribution ({label}):")
    net.plot_weight_distribution()
    net.plot_gradient_distribution()


### 2.d Pengaruh Learning Rate

Menguji learning rate: **0.1, 0.01, 0.001**

In [ ]:
lrs = [0.1, 0.01, 0.001]
input_size = X_train.shape[1]

for lr in lrs:
    print()
    print(f"Training model with Learning Rate {lr}")
    net = FFNN()
    net.add(Layer(input_size, 16, activation='relu', weight_init='random_normal', seed=RANDOM_SEED))
    net.add(Layer(16, 1, activation='sigmoid', weight_init='random_normal', seed=RANDOM_SEED + 1))

    history = fit_ffnn(net, learning_rate=lr)

    plot_loss(history, title=f"LR = {lr}")
    y_pred_val = net.predict(X_val)
    evaluate_model(y_val, y_pred_val, f"Learning Rate {lr} (Validation)")
    print(f"Plotting Weights and Gradients distribution (LR = {lr}):")
    net.plot_weight_distribution()
    net.plot_gradient_distribution()


### 2.e Pengaruh Inisialisasi Bobot

Menguji inisialisasi: **zero, random_uniform, random_normal, xavier, he**.


In [ ]:
initializations = [
    ('zero', {}, 'Zero Initialization'),
    ('random_uniform', {'lower_bound': -0.3, 'upper_bound': 0.3}, 'Random Uniform Initialization'),
    ('random_normal', {'mean': 0.0, 'variance': 0.1}, 'Random Normal Initialization'),
    ('xavier', {}, 'Xavier Initialization'),
    ('he', {}, 'He Initialization'),
]

input_size = X_train.shape[1]

for init_name, init_kwargs, label in initializations:
    print()
    print(f"Training model with {label}")
    net = FFNN()
    net.add(Layer(input_size, 16, activation='relu', weight_init=init_name, seed=RANDOM_SEED, **init_kwargs))
    net.add(Layer(16, 1, activation='sigmoid', weight_init=init_name, seed=RANDOM_SEED + 1, **init_kwargs))

    history = fit_ffnn(net)

    plot_loss(history, title=label)
    y_pred_val = net.predict(X_val)
    evaluate_model(y_val, y_pred_val, f"{label} (Validation)")
    print(f"Plotting Weights and Gradients distribution ({label}):")
    net.plot_weight_distribution()
    net.plot_gradient_distribution()


## 3. Pengaruh Regularisasi

Membandingkan model tanpa regularisasi, dengan L1 (lambda 0.001), dan dengan L2 (lambda 0.001)

In [ ]:
regs = [(0, 0, "No Regularization"), (0.001, 0, "L1 Regularization"), (0, 0.001, "L2 Regularization")]
input_size = X_train.shape[1]

for l1, l2, name in regs:
    print()
    print(f"Training model with {name}")
    net = FFNN()
    net.add(
        Layer(
            input_size,
            16,
            activation='linear',
            weight_init='random_uniform',
            lower_bound=-0.3,
            upper_bound=0.3,
            seed=RANDOM_SEED,
        )
    )
    net.add(
        Layer(
            16,
            16,
            activation='linear',
            weight_init='random_uniform',
            lower_bound=-0.3,
            upper_bound=0.3,
            seed=RANDOM_SEED + 1,
        )
    )
    net.add(
        Layer(
            16,
            1,
            activation='sigmoid',
            weight_init='random_uniform',
            lower_bound=-0.3,
            upper_bound=0.3,
            seed=RANDOM_SEED + 2,
        )
    )

    history = fit_ffnn(net, learning_rate=0.1, l1_lambda=l1, l2_lambda=l2)

    plot_loss(history, title=name)
    y_pred_val = net.predict(X_val)
    evaluate_model(y_val, y_pred_val, f"{name} (Validation)")
    print(f"Plotting Weights and Gradients distribution ({name}):")
    net.plot_weight_distribution()
    net.plot_gradient_distribution()


## 4. Uji Perbandingan dengan Sklearn MLP

Membangun model from scratch terbaik vs `sklearn.neural_network.MLPClassifier`.

In [ ]:
# Final FFNN from Scratch
print("Training FFNN from Scratch (tuned + L1)...")
net_final = FFNN()
net_final.add(
    Layer(
        X_train.shape[1],
        16,
        activation='linear',
        weight_init='random_uniform',
        lower_bound=-0.3,
        upper_bound=0.3,
        seed=RANDOM_SEED,
    )
)
net_final.add(
    Layer(
        16,
        16,
        activation='linear',
        weight_init='random_uniform',
        lower_bound=-0.3,
        upper_bound=0.3,
        seed=RANDOM_SEED + 1,
    )
)
net_final.add(
    Layer(
        16,
        1,
        activation='sigmoid',
        weight_init='random_uniform',
        lower_bound=-0.3,
        upper_bound=0.3,
        seed=RANDOM_SEED + 2,
    )
)
history_final = fit_ffnn(
    net_final,
    learning_rate=0.1,
    epochs=EXPERIMENT_EPOCHS,
    l1_lambda=0.001,
    l2_lambda=0,
    verbose=1,
)

plot_loss(history_final, title="FFNN From Scratch (Tuned + L1) Loss")
y_pred_ffnn_test = net_final.predict(X_test)
evaluate_model(y_test, y_pred_ffnn_test, "FFNN From Scratch Tuned + L1 (Test)")

print("Plotting Weights and Gradients distribution (From Scratch Tuned + L1):")
net_final.plot_weight_distribution()
net_final.plot_gradient_distribution()

print()
print("=" * 40)
print()

# Sklearn MLP 
print("Training Sklearn MLPClassifier")
mlp = MLPClassifier(
    hidden_layer_sizes=(16, 16),
    activation='identity',
    solver='sgd',
    batch_size=BATCH_SIZE,
    learning_rate='constant',
    learning_rate_init=0.1,
    max_iter=EXPERIMENT_EPOCHS,
    random_state=RANDOM_SEED,
)
mlp.fit(X_train, y_train.ravel())

plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, label='Train Loss (Sklearn)')
plt.title("Sklearn MLPClassifier Loss")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

y_pred_sklearn_test = mlp.predict_proba(X_test)[:, 1]
evaluate_model(y_test, y_pred_sklearn_test, "Sklearn MLPClassifier")


5. Save and Load model

In [ ]:
import pickle
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f"../test/model_{timestamp}.pkl"

# Save
with open(model_path, 'wb') as f:
    pickle.dump(net_final, f)
print(f"Model saved to {model_path}")

# Load
with open(model_path, 'rb') as f:
    net_loaded = pickle.load(f)
print("Model loaded successfully")

# Test
y_pred_test = net_loaded.predict(X_test)
evaluate_model(y_test, y_pred_test, "Loaded Model (Test)")
